In [50]:
from dotenv import load_dotenv
load_dotenv()
from langchain_core.tools import tool
from langchain_core.prompts import PromptTemplate
from langchain_community.document_loaders import PyPDFLoader
from langchain.chat_models import init_chat_model
from langchain_mistralai import MistralAIEmbeddings
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [3]:
loader=PyPDFLoader("/home/b1swas/RAG/RagProject/src/Agentic_RAG/medical_report.pdf")
docs=loader.load()
len(docs)

9

In [5]:
splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
splitted_docs=splitter.split_documents(docs)
len(splitted_docs)

26

In [27]:
embedding_model=MistralAIEmbeddings()
vector_store=InMemoryVectorStore.from_documents(
    documents=splitted_docs,
    embedding=embedding_model,
)


In [21]:
same_record=vector_store.similarity_search("Patient name")
same_record[0].page_content

'Report Status    \nFemale\n27 Years:\n:\n:\n:\nAge\nGender\nReported        \nP\n9/7/2025   4:56:00PM\nDR NITIN NAHAR\n474764803\nMs. NIKITA  CHUDHARY:\n:\n:\n:\n:\nName        \nLab No.    \nRef By \nCollected       \nA/c Status Final\n10/7/2025  6:31:50PM\n:Collected at            :Processed at             BHOPAL CC-82\nMr Rachel V John Pata So Vitus John Mig 26 \nGraund,Indrapuri, Phone: 8770817968\n \nLPL-NATIONAL REFERENCE LAB\nNational Reference laboratory, Block E, \nSector 18, Rohini, New Delhi -110085\nTest Report      \nTest Name Results Units Bio. Ref. Interval\nHLA - B27\n(Flow Cytometry)\nHLA-B27, Disease Association   Negative\nInterpretation\n ------------------------------------------------------------\n| RESULT         |         REMARKS                           |\n|                |                                           |\n|----------------|-------------------------------------------|\n| Negative       | No expression of HLA-B27                  |'

In [51]:
@tool
def retriever_tool(query:str):
    """This tool can help you to retrieve the relevant data of the PDF Documents, and then answer according to the details on medical report"""
    print("Tool called",query)
    docs=vector_store.similarity_search(query=query,k=4)
    context=""
    for doc in docs:
        context=doc.page_content+"\n\n"
    return context

In [60]:
retriever_tool("Patient Name")

TypeError: 'StructuredTool' object is not callable

In [52]:
llm_model=init_chat_model(
    model="mistral-small-latest"
)

In [53]:
System_Prompt="""
you are a helpfull assistant that answers questions using retrived context.
Always use the "retriver_tool" for questions requiring external knowledge
"""
agent=create_agent(
    model=llm_model,
    tools=[retriever_tool],
    system_prompt=System_Prompt,
)

In [58]:
query="what is ai?"

response=agent.invoke({"messages":[{'role':'user',"content":query}]})

In [59]:
result=response['messages'][-1].content
print(result)

**Artificial Intelligence (AI)** is a field of computer science that focuses on creating systems or machines capable of performing tasks that typically require human intelligence. These tasks include:

1. **Learning**: AI systems can improve their performance over time by analyzing data and recognizing patterns (e.g., machine learning).
2. **Reasoning**: AI can make decisions or solve problems based on logical rules or algorithms.
3. **Problem-Solving**: AI can analyze complex situations and propose solutions (e.g., game-playing AI like AlphaGo).
4. **Perception**: AI can interpret sensory data, such as images (computer vision) or speech (natural language processing).
5. **Language Understanding**: AI can understand and generate human language (e.g., chatbots, translation tools).
6. **Automation**: AI can perform repetitive tasks without human intervention (e.g., robotic process automation).

AI can be categorized into:
- **Narrow AI (Weak AI)**: Designed for specific tasks (e.g., virt